#Colección y Descripción de Datos — Beneficiarios Estrategia UNIDOS

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, regexp_replace, mean, upper, sum as Fsum

##Carga Inicial de Datos

In [0]:
 
#RUTA ANGE  /Volumes/workspace/proyecto/proyectvolume/dataframes/dataframes/Beneficiarios-Estrategia-UNIDOS.csv
#RUTA NATALIA /Volumes/catalogproyectosparkies/default/volumeproyectosparkies/Beneficiarios-Estrategia-UNIDOS.csv

UNIDOS_PATH = "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/Beneficiarios-Estrategia-UNIDOS.csv"
df_unidos = spark.read.csv(UNIDOS_PATH, header=True, inferSchema=False, sep=",")

display(df_unidos.limit(15))
#para ver toda la tabla: 
# display(df_unidos)

##Descripción de Datos

In [0]:
df_unidos.printSchema()

In [0]:
print("Filas:", df_unidos.count())
print("Columnas:", len(df_unidos.columns))

## Comprensión del significado de atributos clave:

**CodigoFamilia / TipoDocumento**: Identificadores del beneficiario dentro del programa UNIDOS.

**RangoEdad**: Rango etario del beneficiario (06-17, 18-29, 30-49, 50-65, >65).

**NombreMunicipioAtencion / CodigoMunicipioAtencion**: Municipio de Boyacá donde reside el beneficiario.

**PuntajeSISBEN**: Puntaje del beneficiario en el Sistema de Identificación de Potenciales Beneficiarios.

**Estrato**: Estrato socioeconómico (0 a 5; 99 = sin estratificar).

**TipoPoblacion**: Clasificación del beneficiario según ubicación (UNIDOS RURAL, UNIDOS URBANO, UNIDOS U100).

**EstadoBeneficiario**: Indica si el beneficiario está ACTIVO o NO ACTIVO en el programa.

**Logro1 a Logro26**: Estado de cada uno de los 26 logros de superación de pobreza (ALCANZADO, POR ALCANZAR, NO APLICA).

**PE34–PE51**: Indicadores de educación del hogar (asistencia escolar, nivel educativo alcanzado PE42, acceso a programas).

**HE11–HE23**: Indicadores de habitabilidad del hogar (acceso a servicios, condiciones de vivienda).

#Exploración de Datos

In [0]:
#1: EstadoBeneficiario
print("Elemento 1: Conteo por Estado del Beneficiario")
df_unidos.groupBy("EstadoBeneficiario").count().orderBy("count", ascending=False).show()

#2: TipoPoblacion
print("Elemento 2: Conteo por Tipo de Población")
df_unidos.groupBy("TipoPoblacion").count().orderBy("count", ascending=False).show()

#3: 10 municipios con más beneficiarios
print("Elemento 3: Top 10 municipios con más beneficiarios UNIDOS")
df_unidos.groupBy("NombreMunicipioAtencion").count() \
    .orderBy(col("count").desc()).show(10, truncate=False)

#4: RangoEdad
print("Elemento 4: Distribución por Rango de Edad")
df_unidos.groupBy("RangoEdad").count().orderBy("RangoEdad").show()

In [0]:
#5: Beneficiarios por TipoPoblacion
display(
    df_unidos.groupBy("TipoPoblacion")
    .count()
    .orderBy("count", ascending=False)
)

#6: Distribución por RangoEdad
display(
    df_unidos.groupBy("RangoEdad")
    .count()
    .orderBy("RangoEdad")
)

#7: top 8 nivel educativo
display(
    df_unidos.groupBy("PE42")
    .count()
    .orderBy("count", ascending=False)
    .limit(8)
)

#8: Top 10 municipios
display(
    df_unidos.groupBy("NombreMunicipioAtencion")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)

#Reporte de Calidad de Datos

In [0]:
from pyspark.sql.functions import col, count, when, isnan

print("--- REPORTE DE VALORES FALTANTES (NULOS REALES) ---")

expresiones_nulos = []
for columna, tipo in df_unidos.dtypes:
    condicion = col(columna).isNull() | (col(columna) == "") | (col(columna) == "NaN")
    expresiones_nulos.append(count(when(condicion, columna)).alias(columna))

df_unidos.select(expresiones_nulos).show(vertical=True)

In [0]:
print("--- REPORTE DE VALORES 'ND' Y 'Sin Información' (NULOS SEMÁNTICOS) ---")

expresiones_nd = []
for columna, tipo in df_unidos.dtypes:
    condicion = (col(columna) == "ND") | (col(columna) == "Sin Información")
    expresiones_nd.append(count(when(condicion, columna)).alias(columna))

df_unidos.select(expresiones_nd).show(vertical=True)

In [0]:
print("--- REVISIÓN DE DUPLICADOS EXACTOS ---")

total_registros = df_unidos.count()
registros_unicos = df_unidos.dropDuplicates().count()
duplicados = total_registros - registros_unicos

print("Total de registros:", total_registros)
print("Registros únicos:", registros_unicos)
print("Duplicados exactos:", duplicados)

## Técnicas propuestas para tratar valores faltantes:

### Eliminación de columnas no informativas:
Las columnas Pais (84.3% ND), CondicionSexua` (97.5% ND), Etnia (98.9% ND) 
y BeneficiarioSISBEN (80.2% ND) se eliminarán por no aportar información útil al análisis.

### Estandarización de valores inconsistentes:
La columna Parentesco presenta duplicados por diferencias en mayúsculas/minúsculas 
("Jefe" vs "JEFE", "Hijos / Hijastros" vs "HIJOS/ HIJASTROS"). Se unificará con upper().

### Reemplazo semántico:
Los valores "ND" restantes en columnas conservadas como Genero y Parentesco 
se reemplazarán por null para permitir operaciones estadísticas posteriores.

#Filtros, Limpieza y Transformación Inicial

In [0]:
print("--- REVISIÓN DE VALORES DE ESTRATO ---")

df_unidos.groupBy("Estrato") \
    .count() \
    .orderBy("Estrato") \
    .show(truncate=False)

In [0]:
# Filtros 
# 1: Conservar  beneficiarios activos
print(f"Filas antes del filtro: {df_unidos.count()}")
df_unidos = df_unidos.filter(col("EstadoBeneficiario") == "ACTIVO")

print(f"Filas tras filtrar solo ACTIVOS: {df_unidos.count()}")

#2: Eliminar estrato == '99'
df_unidos = df_unidos.filter(col("Estrato") != "99")
print(f"Filas tras eliminar Estrato 99: {df_unidos.count()}")

In [0]:
#Transformaciones
#1: eliminar columnas con más del 80% de ND

cols_eliminar = [
    "Pais", "CondicionSexual", "Etnia", "BeneficiarioSISBEN",
    "NombreDepartamentoAtencion", "CodigoDepartamentoAtencion"
]
df_unidos = df_unidos.drop(*cols_eliminar)
print(f"Columnas tras eliminación: {len(df_unidos.columns)}")

#2: cambiar parentesco a mayúsculas
df_unidos = df_unidos.withColumn("Parentesco", upper(col("Parentesco")))

print("Valores únicos de Parentesco tras estandarización:")
df_unidos.groupBy("Parentesco").count().orderBy("count", ascending=False).show(truncate=False)

#3: Reemplazar ND por null
for c in ["Genero", "Parentesco", "Discapacidad", "EstadoCivil"]:
    df_unidos = df_unidos.withColumn(
        c, when(col(c) == "ND", None).otherwise(col(c))
    )

#4: Crear columna total_logros_alcanzados
logros = [f"Logro{i}" for i in range(1, 27)]

df_unidos = df_unidos.withColumn(
    "total_logros_alcanzados",
    sum([when(col(l) == "ALCANZADO", 1).otherwise(0) for l in logros])
)

df_unidos.select("CodigoFamilia", "NombreMunicipioAtencion", "total_logros_alcanzados").show(5)

#5: Crear columna nivel_educativo_bajo
niveles_bajos = [
    "NINGUNO",
    "PREESCOLAR (PREJARDÍN JARDÍN TRANSICIÓN)",
    "BÁSICA PRIMARIA 1°", "BÁSICA PRIMARIA 2°", "BÁSICA PRIMARIA 3°",
    "BÁSICA PRIMARIA 4°", "BÁSICA PRIMARIA 5°"
]

df_unidos = df_unidos.withColumn(
    "nivel_educativo_bajo",
    when(col("PE42").isin(niveles_bajos), 1).otherwise(0)
)

print("Distribución de nivel_educativo_bajo:")
df_unidos.groupBy("nivel_educativo_bajo").count().show()

In [0]:
print("--- REPORTE DE NULOS TRAS TRANSFORMACIONES ---")

expresiones_finales = []
for columna, tipo in df_unidos.dtypes:
    if tipo == "string":
        condicion = col(columna).isNull() | (col(columna) == "") | (col(columna) == "NaN")
    else:
        condicion = col(columna).isNull()
    expresiones_finales.append(count(when(condicion, columna)).alias(columna))

df_unidos.select(expresiones_finales).show(vertical=True)

print("--- MUESTRA DEL DATASET TRANSFORMADO Y LIMPIO ---")
df_unidos.select(
    "CodigoFamilia", "NombreMunicipioAtencion", "RangoEdad",
    "TipoPoblacion", "Estrato", "PE42",
    "total_logros_alcanzados", "nivel_educativo_bajo"
).show(5, truncate=False)

print(f"\nFilas finales: {df_unidos.count()}")
print(f"Columnas finales: {len(df_unidos.columns)}")

In [0]:
df_unidos.groupBy("NombreMunicipioAtencion").count().orderBy("count", ascending=False).show(truncate=False)

In [0]:
df_unidos.groupBy("").count().orderBy("count", ascending=False).show(truncate=False)

In [0]:
display(df_unidos)